# Bharat Praman — Fine-Tune `Qwen2.5-Coder-3B-Instruct` for Text-to-Cypher (QLoRA)

**Runtime:** In Google Colab, select **Runtime -> Change runtime type -> T4 GPU**.

This notebook:
1. Installs QLoRA dependencies (`transformers`, `peft`, `trl`, `bitsandbytes`, `accelerate`, `datasets`).
2. Lets you upload `data/finetune_train_verified.jsonl` and `data/finetune_val_verified.jsonl`.
3. Fine-tunes `Qwen/Qwen2.5-Coder-3B-Instruct` in 4-bit NF4 QLoRA.
4. Evaluates JSON structure & Cypher generation on the validation split.
5. Merges the LoRA adapter and converts it to **GGUF (`Q4_K_M`)** + generates an **Ollama `Modelfile`** so you can run it locally on Windows CPU with zero GPU needed.

In [ ]:
!nvidia-smi
!pip install -q -U transformers datasets accelerate peft bitsandbytes trl huggingface_hub

In [ ]:
from google.colab import files
import os

os.makedirs("data", exist_ok=True)
print("Upload finetune_train_verified.jsonl and finetune_val_verified.jsonl:")
uploaded = files.upload()
for fname, content in uploaded.items():
    dest = os.path.join("data", os.path.basename(fname))
    with open(dest, "wb") as f:
        f.write(content)
    print("Saved:", dest)

In [ ]:
import json
import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

BASE_MODEL = "Qwen/Qwen2.5-Coder-3B-Instruct"
ADAPTER_DIR = "outputs/bharat-praman-cypher-lora"

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = load_jsonl("data/finetune_train_verified.jsonl")
val_rows = load_jsonl("data/finetune_val_verified.jsonl")
print(f"Loaded {len(train_rows)} train / {len(val_rows)} val verified examples")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(row):
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False)}

train_ds = Dataset.from_list(train_rows).map(format_example)
val_ds = Dataset.from_list(val_rows).map(format_example)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

sft_config = SFTConfig(
    output_dir=ADAPTER_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=False,
    bf16=False,
    max_length=2048,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora_config,
    args=sft_config,
)
trainer.train()
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved best LoRA adapter to:", ADAPTER_DIR)

In [ ]:
# Evaluate JSON & Cypher accuracy on validation split
model.eval()
json_ok = 0
type_ok = 0
exact_cypher = 0

for i, row in enumerate(val_rows):
    prompt_msgs = row["messages"][:2]
    gold = json.loads(row["messages"][2]["content"])
    text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=320, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    try:
        pred = json.loads(gen)
        json_ok += 1
        if pred.get("result_type") == gold.get("result_type"):
            type_ok += 1
        if pred.get("cypher", "").strip() == gold.get("cypher", "").strip():
            exact_cypher += 1
    except Exception as e:
        if i < 5:
            print(f"Val #{i} invalid JSON:", gen[:160])

n = len(val_rows)
print(f"Validation results (n={n}):")
print(f"  Valid JSON:          {json_ok}/{n} ({json_ok/n:.1%})")
print(f"  Result-type match:   {type_ok}/{n} ({type_ok/n:.1%})")
print(f"  Exact Cypher match:  {exact_cypher}/{n} ({exact_cypher/n:.1%})")

In [ ]:
# Merge LoRA into FP16 model and convert to Q4_K_M GGUF for local Ollama on Windows
import gc
from peft import PeftModel

del model, trainer
gc.collect()
torch.cuda.empty_cache()

MERGED_DIR = "outputs/bharat-praman-cypher-merged"
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="auto")
merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print("Merged model saved to:", MERGED_DIR)

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q gguf sentencepiece protobuf
!cmake -B llama.cpp/build llama.cpp -DGGML_CUDA=OFF && cmake --build llama.cpp/build --target llama-quantize -j 4
!python llama.cpp/convert_hf_to_gguf.py outputs/bharat-praman-cypher-merged --outfile bharat-praman-cypher-f16.gguf --outtype f16
!./llama.cpp/build/bin/llama-quantize bharat-praman-cypher-f16.gguf bharat-praman-cypher-q4_k_m.gguf Q4_K_M

modelfile = '''FROM ./bharat-praman-cypher-q4_k_m.gguf
PARAMETER temperature 0
PARAMETER num_ctx 4096
'''
with open("Modelfile", "w") as f:
    f.write(modelfile)

!ls -lh bharat-praman-cypher-q4_k_m.gguf Modelfile

In [ ]:
# Download the quantized GGUF (~2 GB) + Modelfile to use locally with Ollama
files.download("bharat-praman-cypher-q4_k_m.gguf")
files.download("Modelfile")